# 02: IQPEのビット数と測定精度の相関
### (Correlation between the number of bits in QIPE and measurement accuracy)

---

## 1. Thesis: 科学的問いと仮説
量子位相推定においては、位相を高精度で推定することが重要になります。標準的な量子位相推定（QPE）は多数の量子ビットを必要としますが、位相のみが関心対象であり固有状態の再利用が可能な場合、より少ない量子ビットで同等の情報を得られる可能性があります。

**問い**: 補助量子ビットを1量子ビットに制限した場合でも、位相情報を測定確率として逐次的に読み出し、高精度で位相を推定することは可能なのか？また、その推定精度は反復回数にどのように依存するのか？

**仮説**: 反復量子位相推定（Iterative Quantum Phase Estimation, IPE）の標準的実装であるLSB優先方式は、位相 $θ$ を二進展開 $θ=2π(0.b_1​b_2​⋯b_n​)$ で表現した際に、最下位ビット $b_n$ から順番に決定していく方式であり、この手法では補助量子ビット1つのみを用いて $n$ ビットの位相情報を抽出でき、推定誤差はおおよそ使用ビット数 $n$ に対しておおよそ $2π / 2^n$ に比例して減少することが期待される。

## 2. Theoretical Background (理論的背景)

### 2.1 標準的量子位相推定 (QPE) と反復量子位相推定 (IQPE) の比較
標準的な量子位相推定（QPE: Quantum Phase Estimation）は、求めたい位相の精度（ビット数 $n$）と同じ数の補助量子ビットを必要とします。これは、位相情報を一度の測定でエンコードし、逆量子フーリエ変換 (IQFT) を用いて一気に読み出すためです。 しかし、現実の量子コンピュータ（特にNISQデバイス）では、多数の量子ビットをコヒーレンスを保ったまま操作することは困難です。

そこで、固有状態を繰り返し準備・再利用できるという前提のもと、補助量子ビットを「1つだけ」使い、1ビットずつ順番に位相を決定していく手法が反復量子位相推定 (IQPE: Iterative Quantum Phase Estimation) です。

### 2.2 位相の二進展開とスケーリング
推定したい未知の位相 $\theta$ を、2進数を用いて小数展開します。

$$
\theta = 2\pi(0.b_1 b_2 \dots b_n) = 2\pi \left( \frac{b_1}{2} + \frac{b_2}{2^2} + \dots + \frac{b_n}{2^n} \right)
$$

ここで、$b_i \in {0, 1}$ は各桁のビット値です。

IQPEにおいて極めて重要なのが「位相のスケーリング（増幅）」です。 位相 $\theta$ に $2^k$ を掛ける（位相回転を $2^k$ 回繰り返す）と、上位 $k$ ビット分が整数（$2\pi$の倍数）となって測定に影響しなくなり、小数部分がシフトします。

$$ 2^{n-1}\theta = 2\pi(b_1 \dots b_{n-1}.b_n) \equiv 2\pi(0.b_n) \pmod{2\pi} $$

これにより、非常に微小な最下位ビット $b_n$ の位相寄与を、測定可能な半周（$\pi$）レベルの大きな位相差へと増幅して取り出すことが可能になります。

### 2.3 LSB優先 (Least Significant Bit First) アプローチ
IQPEは、標準的にLSB（最下位ビット $b_n$）から順番に決定していきます。

**ステップ1 (LSBの決定)** :  
位相を $2^{n-1}$ 倍にスケーリングすると、上位ビットの影響が消え、位相は $2\pi(0.b_n)$ のみに依存します。これを $H$ ゲートで干渉させて測定すれば、$b_n$ が 0 か 1 かを確定できます。

**ステップ2 (下位から2番目の決定)** :  
次に位相を $2^{n-2}$ 倍にスケーリングします。位相は $2\pi(0.b_{n-1}b_n)$ となります。 ここで、ステップ1で既に決定した $b_n$ の情報を用いて、逆回転（位相補正）$R_Z(-2\pi \cdot 0.0b_n)$ をかけます。 すると $b_n$ の影響が打ち消され、$2\pi(0.b_{n-1})$ だけが残り、$b_{n-1}$ を正確に測定できるようになります。

**繰り返し** :  
常に「これまでに決定した下位ビット群」の寄与をソフトウェア的に逆回転でキャンセルしながら、一つ上位のビットを決定していきます。最後にMSB（最上位ビット $b_1$）を決定して完了です。

### 2.4 IQPEの基本回路構成 (H / RZ / 測定)
1ビットの位相情報の抽出方法の基本構造は、「01_Physics_of_Phase.ipynb」 で学んだラムゼー干渉計と同じです。

**初期化**:  
$H$ ゲートで補助ビットを $|+\rangle$ 状態にする。

**位相回転** :  
対象の系（固有状態）との相互作用、あるいは外部から $R_Z(\theta \cdot 2^i)$ をかけ、特定のビットにフォーカスした位相情報をエンコードする。

**位相補正** :  
既知の下位ビット群の寄与を $R_Z(- \text{correction})$ によって打ち消す。

**干渉と測定** :  
再び $H$ ゲートをかけて $Z$ 基底へ戻し、0または1を測定する。
これを指定したビット精度 $n$ の回数だけ、スケール $i$ を $n-1$ から $0$ まで減らしながらループ実行します。

## 3. Implementation (実装)
外部ライブラリ "Qiskit" から `AerSimulator`、`QuantumCircuit` 、`transpile` をインポートします。

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import numpy as np

def iterative_phase_estimation(true_theta: float, num_bits: int = 6) -> float:
    """反復量子位相推定(IQPE)アルゴリズムを用いて位相を推定します.

    量子位相推定の省リソース版.
    1量子ビットのみを使用して位相を最下位ビットから順に決定していきます.

    Args:
        true_theta (float): 推定対象の真の位相[rad] (0 ~ 2π)
        num_bits (int, optional): 推定精度(ビット数). Defaults to 6.
            精度 ≈ 2π / 2^num_bits (例: 6ビット → 約0.098 rad)

    Returns:
        float: 推定された位相[rad]

    Algorithm:
        各ビットiについて(最下位ビットから):
        1. |+⟩状態を準備
        2. true_theta × 2^i の位相回転を適用
        3. 既に決定したビットによる補正(-estimated_phase × 2^i)
        4. 測定により現在のビット値を決定
        5. 結果に応じて推定位相を更新

    Example:
        >>> # π/4 (約0.785 rad)を推定
        >>> iterative_phase_estimation(np.pi/4, num_bits=6)
        --- IPE algorithm (Bits: 6) ---
            修正した結果: 0.7854
    """
    print(f"\n--- IPE algorithm (Bits: {num_bits}) ---")

    estimated_phase = 0.0
    for i in reversed(range(num_bits)):
        scaling = 2**i
        qc = QuantumCircuit(1, 1)
        qc.h(0)  # H（重ね合わせ）
        qc.rz(true_theta * scaling, 0)  # RZ（位相回転）
        qc.rz(-estimated_phase * scaling, 0)  # RZ (既知位相の打ち消し)
        qc.h(0)  # H（干渉）
        qc.measure(0, 0)  # 測定

        # --- 実行 ---
        sim = AerSimulator()
        simulation_result = sim.run(transpile(qc, sim), shots=1).result()
        measured_bit = int(list(simulation_result.get_counts().keys())[0])

        # 位相の更新
        if measured_bit == 1:
            estimated_phase += (1 / 2 ** (i + 1)) * (2 * np.pi)

    print(f" 修正した結果: {estimated_phase:.4f}")
    return estimated_phase

## 4. Visualization & Analysis (可視化と解析)

## 5. Conclusion & Future Work (結論と展望)